In [70]:
import geopandas as gpd
import pandas as pd

In [71]:
current_flu_year = 2026
cur_flu_yr_2dig = str(current_flu_year)[-2:]
flu_csv_path = "C:/Users/JKolberg/OneDrive - PSRC/GIS - Projects/FLU/Zoning_2026_d2.csv"

old_flu_year = 2019
old_flu_yr_2dig = str(old_flu_year)[-2:]
old_flu_shp_path = "W:/gis/projects/compplan_zoning/flu19_reviewed.shp"

In [72]:
# get old xwalk for old FLU descriptions
old_xwalk_path = "J:/Staff/Christy/usim-baseyear/flu/Full_FLU_Master_Corres_File.xlsx"
old_xwalk = pd.read_excel(old_xwalk_path, sheet_name='Full Master FLU Corres File')[['FLUadj_Key','FLUadj_Definition']].dropna()

In [73]:
flu_table = pd.read_csv(flu_csv_path, encoding='latin-1')
flu_table = flu_table.drop(columns=[c for c in flu_table.columns if 'Unnamed' in c]).add_suffix(f'_{cur_flu_yr_2dig}')

In [74]:
old_flu = gpd.read_file(old_flu_shp_path)
old_flu_table = (
    old_flu
    .drop(columns=['geometry'])
    .copy()
    .add_suffix(f'_{old_flu_yr_2dig}')
    .merge(old_xwalk, left_on=f'Juris_zn_{old_flu_yr_2dig}', right_on='FLUadj_Key', how='left')
)
old_flu_shp = old_flu[['Juris_zn','geometry']].copy()

c:\Users\JKolberg\PythonProjects\PSRC\urbansim-baseyear-prep\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\JKolberg\PythonProjects\PSRC\urbansim-baseyear-prep\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 285 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(
c:\Users\JKolberg\PythonProjects\PSRC\urbansim-baseyear-prep\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 1768 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


In [75]:
old_flu_table.columns

Index(['Jurisdicti_19', 'Juris_zn_19', 'Zone_adj_19', 'Juris_zn_1_19',
       'Res_Use_19', 'Comm_Use_19', 'Office_Use_19', 'Indust_Use_19',
       'Mixed_Use_19', 'Public_Use_19', 'PUD_Use_19', 'MaxDU_Res_19',
       'MaxFAR_Com_19', 'MaxFAR_Off_19', 'MaxFAR_Ind_19', 'MaxDU_Mixe_19',
       'MaxFAR_Mix_19', 'MaxHt_Res_19', 'MaxHt_Mixe_19', 'MaxHt_Comm_19',
       'MaxHt_Offi_19', 'MaxHt_Indu_19', 'FLUadj_Key', 'FLUadj_Definition'],
      dtype='str')

In [76]:
flu_table.columns

Index(['Juris_26', 'Zone_26', 'juris_zn_26', 'Definition_26', 'Bonus_avail_26',
       'Bonus_included_26', 'Res_Use_26', 'Comm_Use_26', 'Office_Use_26',
       'Indust_Use_26', 'Mixed_Use_26', 'Public_Use_26', 'PUD_Use_26',
       'ResDU_lot_26', 'MinDU_Res_26', 'MinDU_Comm_26', 'MinDU_Office_26',
       'MinDU_Indust_26', 'MinDU_Mixed_26', 'MaxDU_Res_26', 'MaxDU_Comm_26',
       'MaxDU_Office_26', 'MaxDU_Indust_26', 'MaxDU_Mixed_26', 'MinFAR_Res_26',
       'MinFAR_Comm_26', 'MinFAR_Office_26', 'MinFAR_Indust_26',
       'MinFAR_Mixed_26', 'MaxFAR_Res_26', 'MaxFAR_Comm_26',
       'MaxFAR_Office_26', 'MaxFAR_Indust_26', 'MaxFAR_Mixed_26',
       'MaxHt_Res_26', 'MaxHt_Comm_26', 'MaxHt_Office_26', 'MaxHt_Indust_26',
       'MaxHt_Mixed_26', 'LC_Res_26', 'LC_Comm_26', 'LC_Office_26',
       'LC_Indust_26', 'LC_Mixed_26', 'rural_26', 'ADU notes_26'],
      dtype='str')

In [77]:
old_to_new_col_mapping = {
    'Jurisdicti_19': 'Juris_26',
    'Juris_zn_19': 'juris_zn_26',
    'FLUadj_Definition': 'Definition_26'
}

In [78]:
from difflib import SequenceMatcher
import re

def normalize_zone(s):
    """Normalize a zone string for near-matching: lowercase, strip, collapse separators."""
    s = str(s).strip().lower()
    s = re.sub(r'[-_\s,]+', '', s)  # remove dashes, underscores, spaces, commas
    return s

# --- Build unique zone lists per jurisdiction for old (19) and new (26) ---

old_zones = (
    old_flu_table[['Jurisdicti_19', 'Juris_zn_19', 'FLUadj_Definition']]
    .drop_duplicates()
    .rename(columns={'Jurisdicti_19': 'jurisdiction', 'Juris_zn_19': 'zone_19', 'FLUadj_Definition': 'desc_19'})
)

new_zones = (
    flu_table[['Juris_26', 'juris_zn_26', 'Definition_26']]
    .drop_duplicates()
    .rename(columns={'Juris_26': 'jurisdiction', 'juris_zn_26': 'zone_26', 'Definition_26': 'desc_26'})
    .drop_duplicates(subset=['jurisdiction', 'zone_26'], keep='first')
)

print(f"Old zones: {len(old_zones)} unique rows")
print(f"New zones: {len(new_zones)} unique rows")

Old zones: 1919 unique rows
New zones: 1644 unique rows


In [79]:
crosswalk_rows = []

# Normalize jurisdiction names for grouping (handles "Black Diamond" vs "BlackDiamond" etc.)
new_zones['jurisdiction_norm'] = new_zones['jurisdiction'].apply(normalize_zone)
old_zones['jurisdiction_norm'] = old_zones['jurisdiction'].apply(normalize_zone)

# Iterate over new (26) jurisdictions to ensure every new zone is in the crosswalk
for juris_norm in new_zones['jurisdiction_norm'].unique():
    new_j = new_zones[new_zones['jurisdiction_norm'] == juris_norm].copy()
    old_j = old_zones[old_zones['jurisdiction_norm'] == juris_norm].copy()
    juris = new_j['jurisdiction'].iloc[0]  # use the new jurisdiction name as canonical
    
    matched_old = set()
    matched_new = set()
    
    # --- Pass 1: Exact match on Juris_zn ---
    for ni, nrow in new_j.iterrows():
        for oi, orow in old_j.iterrows():
            if oi in matched_old:
                continue
            if str(nrow['zone_26']).strip() == str(orow['zone_19']).strip():
                crosswalk_rows.append({
                    'jurisdiction': juris,
                    'zone_19': orow['zone_19'], 'desc_19': orow['desc_19'],
                    'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                    'match_type': 'exact', 'confidence': 1.0
                })
                matched_old.add(oi)
                matched_new.add(ni)
                break

    # --- Pass 2: Normalized near-match on Juris_zn (dashes/underscores/case) ---
    remaining_new = new_j[~new_j.index.isin(matched_new)]
    remaining_old = old_j[~old_j.index.isin(matched_old)]
    
    for ni, nrow in remaining_new.iterrows():
        norm_new = normalize_zone(nrow['zone_26'])
        for oi, orow in remaining_old.iterrows():
            if oi in matched_old:
                continue
            if norm_new == normalize_zone(orow['zone_19']):
                crosswalk_rows.append({
                    'jurisdiction': juris,
                    'zone_19': orow['zone_19'], 'desc_19': orow['desc_19'],
                    'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                    'match_type': 'near_match', 'confidence': 0.8
                })
                matched_old.add(oi)
                matched_new.add(ni)
                break

    # --- Pass 3: Fuzzy match on description text (for remaining new zones) ---
    remaining_new = new_j[~new_j.index.isin(matched_new)]
    remaining_old = old_j[~old_j.index.isin(matched_old)]
    
    for ni, nrow in remaining_new.iterrows():
        best_score = 0.0
        best_oi = None
        best_orow = None
        new_desc = str(nrow['desc_26']).strip().lower() if pd.notna(nrow['desc_26']) else ''
        
        for oi, orow in remaining_old.iterrows():
            if oi in matched_old:
                continue
            old_desc = str(orow['desc_19']).strip().lower() if pd.notna(orow['desc_19']) else ''
            
            # combine zone-name similarity and description similarity
            zone_sim = SequenceMatcher(None, normalize_zone(nrow['zone_26']), normalize_zone(orow['zone_19'])).ratio()
            desc_sim = SequenceMatcher(None, new_desc, old_desc).ratio() if new_desc and old_desc else 0.0
            score = 0.4 * zone_sim + 0.6 * desc_sim  # weight description higher
            
            if score > best_score:
                best_score = score
                best_oi = oi
                best_orow = orow
        
        if best_score >= 0.4 and best_orow is not None:
            crosswalk_rows.append({
                'jurisdiction': juris,
                'zone_19': best_orow['zone_19'], 'desc_19': best_orow['desc_19'],
                'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                'match_type': 'fuzzy', 'confidence': round(best_score, 3)
            })
            matched_old.add(best_oi)
        else:
            # New zone with no old match — still included for completeness
            crosswalk_rows.append({
                'jurisdiction': juris,
                'zone_19': None, 'desc_19': None,
                'zone_26': nrow['zone_26'], 'desc_26': nrow['desc_26'],
                'match_type': 'new_zone', 'confidence': 0.0
            })

crosswalk = pd.DataFrame(crosswalk_rows)
print(crosswalk['match_type'].value_counts())
print(f"\nTotal crosswalk rows: {len(crosswalk)}")
print(f"Total new zones: {len(new_zones)} — all accounted for: {len(crosswalk) == len(new_zones)}")

match_type
exact         742
new_zone      585
near_match    218
fuzzy          99
Name: count, dtype: int64

Total crosswalk rows: 1644
Total new zones: 1644 — all accounted for: True


In [80]:
# Flag rows needing manual review: fuzzy matches and new zones with no old equivalent
crosswalk['needs_review'] = crosswalk['match_type'].isin(['fuzzy', 'new_zone'])

# Add a column for manual override
crosswalk['manual_match'] = ''

# Show summary
print("=== Match Summary ===")
print(crosswalk.groupby('match_type')['confidence'].describe()[['count','mean','min','max']])
print(f"\nRows needing manual review: {crosswalk['needs_review'].sum()}")
print(f"Rows auto-matched (exact + near): {(~crosswalk['needs_review']).sum()}")

=== Match Summary ===
            count      mean    min    max
match_type                               
exact       742.0  1.000000  1.000  1.000
fuzzy        99.0  0.451525  0.401  0.921
near_match  218.0  0.800000  0.800  0.800
new_zone    585.0  0.000000  0.000  0.000

Rows needing manual review: 684
Rows auto-matched (exact + near): 960


In [81]:
# View only the rows that need manual review, sorted by jurisdiction then confidence
review_df = (
    crosswalk[crosswalk['needs_review']]
    .sort_values(['jurisdiction', 'match_type', 'confidence'])
    [['jurisdiction', 'zone_19', 'desc_19', 'zone_26', 'desc_26', 'match_type', 'confidence', 'manual_match']]
)
print(f"Rows for manual review: {len(review_df)}")
review_df

Rows for manual review: 684


,jurisdiction,zone_19,desc_19,zone_26,desc_26,match_type,confidence,manual_match
0,Algona,NaN,NaN,Algona_C-1,The C-1 mixed use commercial district is inten...,new_zone,0.0,
1,Algona,NaN,NaN,Algona_C-2,The C-2 general commercial district is intende...,new_zone,0.0,
2,Algona,NaN,NaN,Algona_C-3,The C-3 heavy commercial district is intended ...,new_zone,0.0,
3,Algona,NaN,NaN,Algona_M-1,Light industrial zones are intended for light ...,new_zone,0.0,
4,Algona,NaN,NaN,Algona_R-L,The R-L low density residential district is in...,new_zone,0.0,
...,...,...,...,...,...,...,...,...
1594,Woodway,NaN,NaN,Woodway_R-43,The primary function of the residential R-43 z...,new_zone,0.0,
1595,Woodway,NaN,NaN,Woodway_R-87,The primary function of the residential R-87 z...,new_zone,0.0,
1596,Woodway,NaN,NaN,Woodway_UV,The purpose of the urban village (UV) zone is ...,new_zone,0.0,
1597,Yarrow Point,NaN,NaN,Yarrow Point_R-12,The purpose of this title is to regulate the u...,new_zone,0.0,


In [82]:
# Read manual adjustments from Excel and apply to crosswalk
manual = pd.read_excel('flu_crosswalk_19_to_26.xlsx')

# Apply manual overrides where manual_match is non-empty
manual_edits = manual[manual['manual_match'].notna() & (manual['manual_match'].astype(str).str.strip() != '')]

print(f"Found {len(manual_edits)} manual adjustments")

for _, row in manual_edits.iterrows():
    mask = (crosswalk['jurisdiction'] == row['jurisdiction']) & (crosswalk['zone_26'] == row['zone_26'])
    if mask.any():
        manual_val = str(row['manual_match']).strip()
        crosswalk.loc[mask, 'manual_match'] = manual_val
        # Look up the old zone info from old_zones
        old_match = old_zones[old_zones['zone_19'] == manual_val]
        if not old_match.empty:
            crosswalk.loc[mask, 'zone_19'] = manual_val
            crosswalk.loc[mask, 'desc_19'] = old_match.iloc[0]['desc_19']
            crosswalk.loc[mask, 'match_type'] = 'manual'
            crosswalk.loc[mask, 'confidence'] = 1.0
            crosswalk.loc[mask, 'needs_review'] = False
        print(f"  Updated: {row['jurisdiction']} / {row['zone_26']} -> {manual_val}")
    else:
        print(f"  WARNING: No match in crosswalk for {row['jurisdiction']} / {row['zone_26']}")

print(f"\n=== Updated Match Summary ===")
print(crosswalk['match_type'].value_counts())
print(f"Rows still needing review: {crosswalk['needs_review'].sum()}")

Found 42 manual adjustments
  Updated: Algona / Algona_C-1 -> Algona_MIXEDUSECOMMERCIAL
  Updated: Algona / Algona_C-2 -> Algona_GENERALCOMMERCIAL
  Updated: Algona / Algona_C-3 -> Algona_HEAVYCOMMERCIAL
  Updated: Algona / Algona_M-1 -> Algona_LIGHTINDUSTRIAL
  Updated: Algona / Algona_R-L -> Algona_LOWDENSITYRESIDENTIAL
  Updated: Algona / Algona_R-M -> Algona_MEDIUMDENSITYRESIDENTIAL
  Updated: Arlington / Arlington_AP-ISZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-ITZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-OSZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-RPZ -> Arlington_AF
  Updated: Arlington / Arlington_AP-SSZ -> Arlington_AF
  Updated: Arlington / Arlington_GC_MXD -> Arlington_GC
  Updated: Arlington / Arlington_HC_MXD -> Arlington_HC
  Updated: Arlington / Arlington_RHC -> Arlington_RHD
  Updated: Arlington / Arlington_RMC -> Arlington_RMD
  Updated: Arlington / Arlington_P?SP -> Arlington_P/SP
  Updated: Auburn / Auburn_OS -> Auburn_OPEN SPACE
  U

In [83]:
# Export: full crosswalk, review-only subset, and old zones to CSV
crosswalk.to_csv('flu_crosswalk_19_to_26.csv', index=False)
#review_df.to_csv('flu_crosswalk_19_to_26_review.csv', index=False)
old_zones.to_csv('old_zones_19.csv', index=False)
print("Saved flu_crosswalk_19_to_26.csv (full), flu_crosswalk_19_to_26_review.csv (manual review only), and old_zones_19.csv")

Saved flu_crosswalk_19_to_26.csv (full), flu_crosswalk_19_to_26_review.csv (manual review only), and old_zones_19.csv
